# Collin County expansion · status and next processing stage

**Current status:** official county boundary and all 16,324 Census blocks acquired. Seventeen source geometries repaired. The website can show the county's block geography, but only the original four-ZIP pilot has weighted accessibility results.

A county GeoAI tile plan contains 8,085 inference tiles, compared with 505 in the pilot. Planning is not inference completion. Existing pilot scores must not be extrapolated to the remaining county.


In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'scripts'))
report=json.loads((ROOT/'output/collin_county/expansion_status.json').read_text())
display(pd.Series({k:report[k] for k in ['census_blocks','county_area_km2','pilot_area_km2','planned_geoai_tiles','block_geometry_valid']},name='County preparation'))
print(report['status'])


census_blocks                 16324
county_area_km2         2295.462472
pilot_area_km2           124.119504
planned_geoai_tiles            8085
block_geometry_valid           True
Name: County preparation, dtype: object

County geography acquired; full county weighted network, environmental feature coverage and destination inventory NOT yet acquired or scored.


## County acquisition and inference

The following cells are deliberately opt-in for the larger processing stage. They are not needed to run the deployed four-ZIP category selector.

1. Acquire a county walking network with a routing buffer and retain disconnected components.
2. Acquire county POIs, including category and access metadata, plus destinations near the boundary.
3. Extend environmental inputs. Choose a consistent reference for county traffic/temperature indices; do not mix city-specific ranks silently.
4. Run county imagery inference using the configured boundary and separate output folder.
5. Adapt aggregation to the county block inventory, validate missingness/access, and rerun category routing. Current summarizers and scoring scripts still use the pilot's source paths and require explicit county inputs.

A downloaded street graph is not yet a validated sidewalk network. Geometry and block membership alone do not imply known friction weights.


In [2]:
DOWNLOAD_COUNTY_NETWORK=False
if DOWNLOAD_COUNTY_NETWORK:
    import osmnx as ox
    import geopandas as gpd
    county=gpd.read_file(ROOT/'output/collin_county/county_boundary.geojson').to_crs(26914)
    routing_area=gpd.GeoSeries([county.geometry.union_all().buffer(2000)],crs=26914).to_crs(4326).iloc[0]
    ox.settings.use_cache=True
    ox.settings.cache_folder=str(ROOT/'data/raw/collin_county_osm_cache')
    ox.settings.requests_timeout=300
    graph=ox.graph_from_polygon(routing_area,network_type='walk',simplify=True,retain_all=True,truncate_by_edge=True)
    ox.save_graphml(graph,ROOT/'output/collin_county/walking_network.graphml')
    print('Acquired graph:',len(graph),'nodes;',graph.number_of_edges(),'directed edges. Environmental weights not yet assigned.')


In [3]:
RUN_COUNTY_IMAGERY_INFERENCE=False
if RUN_COUNTY_IMAGERY_INFERENCE:
    import os,subprocess
    task_environment=os.environ.copy()
    task_environment['EMPATH_SCOPE_FILE']=str(ROOT/'output/collin_county/county_boundary.geojson')
    task_environment['EMPATH_GEOAI_OUTPUT']=str(ROOT/'output/collin_county/geoai')
    subprocess.run([str(ROOT/'.venv/bin/python'),str(ROOT/'scripts/full_scope_geoai_api.py')],env=task_environment,check=True)
    print('Imagery inference finished. County summaries, environmental joins and routing remain separate stages.')
